# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all data elements by their `@id` fields as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. All references use Croissant `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("\n=== DATASET METADATA ===")
print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Authors: {[a['@id'] for a in getattr(metadata, 'author', [])]}")

## 2. Data Overview
Review the available record sets, fields, and their `@id` fields.

We use `dataset.record_sets` to enumerate all record sets (tables/files), and for each, display their fields and columns using their `@id`s.

In [ ]:
print("\n=== RECORD SETS AVAILABLE ===\n")

all_record_sets = list(dataset.record_sets)

for rs in all_record_sets:
    print(f"Record set name: {rs.name}")
    print(f"@id: {rs['@id']}")
    print(f"Description: {getattr(rs, 'description', '')}")
    if hasattr(rs, 'fields'):
        print("  Fields/columns (@id):")
        for field in rs.fields:
            print(f"    - {field['@id']} (name: {field.name}, type: {getattr(field, 'data_type', '')})")
    print("-")

if not all_record_sets:
    print("(No record sets are explicitly defined in this dataset's Croissant schema)")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis.

All references use the record set and field/column `@id` from the previous overview. Modify the `record_set_id` variable to choose a record set for extraction.


In [ ]:
# Get all available record set @id's from metadata
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Found record set @id's:", record_set_ids)

# If no record sets available, skip the remainder of this cell
if record_set_ids:
    dataframes = {}
    for record_set_id in record_set_ids:
        print(f"\nLoading records from record set '@id': {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. DataFrame preview:")
        print(df.head())
    # Example: select the first record set for further processing
    selected_record_set_id = record_set_ids[0]
    print("\nColumns in DataFrame:")
    print(dataframes[selected_record_set_id].columns.tolist())
else:
    print("No record sets to extract records from.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing: filtering, normalizing numeric fields, outlier removal, and grouping, referencing all columns by their `@id` fields. Example code assumes at least one numeric field in the loaded DataFrame.

In [ ]:
import numpy as np

if record_set_ids:
    df = dataframes[selected_record_set_id]
    
    print(f"Available columns in '{selected_record_set_id}' record set:")
    print(df.columns.tolist())
    # Attempt to auto-detect a numeric field by pandas dtype
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_field_candidates:
        print("No numeric fields detected; cannot perform numeric EDA.")
    else:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.9) # Example: use 90th percentile as outlier threshold
        filtered_df = df[df[numeric_field_id] < threshold]
        print(f"Filtered {len(df)-len(filtered_df)} outlier records with {numeric_field_id} >= {threshold:.2f}.")
        
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nFirst 5 normalized values for {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Attempt to find a group field (object/Categorical or string type)
        group_field_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped by {group_field_id} (showing mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("Skip EDA: No data frames loaded from record sets.")

## 5. Visualization
Visualize the distribution of selected fields using matplotlib or seaborn. Reference all columns by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    # If group_field_id defined, show mean values per group
    if 'group_field_id' in locals():
        plt.figure(figsize=(8,4))
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=group_means.values, y=group_means.index)
        plt.xlabel(f"Mean of {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Visualization skipped: No numeric field available.")

## 6. Conclusion

This notebook demonstrated how to load metadata and records from a Croissant dataset, referencing all record sets, fields, and columns by their `@id`. We explored the data structure programmatically, performed simple normalization and grouping, and visualized numeric field distributions—all using the [mlcroissant](https://github.com/mlcommons/croissant) library.

**Key Takeaways:**
- All entities referenced by `@id` support reliable programmatic access and reproducibility.
- Croissant schemas enable introspection of dataset structure and support robust data loading workflows.
- This FAIR² dataset allows for advanced modeling and equity-focused insights into rangeland knowledge adoption.
